# PZ data challenge: task set 1

**Task set 1: representative training.** The training and test sets are drawn from the same distribution (i < 23.5).

This notebook only trains. For each of the four configurations of the task set (cardinal and flagship, 1yr and 10yr) it trains the model on the training file, holding out `VAL_FRAC` of it (split by `SEED`), and saves it as a single `.pkl` under its official name in `OUT_DIR` (the subtask 2 file, via `znn.save_model_file`), shared by all four task-set notebooks. Everything that uses the models, the test-set p(z)s and the scores on the held-out objects, is in `pz_challenge_results.ipynb`, which predicts from these files.

The model is trained with `znn.train_model`, on the photometry as given. It is a pair of `znn.build_model` networks (residual squeeze-excitation nets), after the moment networks of Jeffrey & Wandelt 2020: F predicts log(1 + z), then G, trained on F's residuals with F fixed, predicts their scale, as a Student-t with `NU` degrees of freedom. The p(z) (in `pz_challenge_results.ipynb`) is that Student-t around F's prediction. With `PRETRAINING = "popcosmos"`, F starts from its pop-cosmos pre-trained weights shipped with znn, every layer trainable; G always starts from scratch. A configuration whose model file is already in `OUT_DIR` is not retrained; delete the file to retrain.

Run this from the repo root with the `qp` conda env as the kernel. The filter curves are the ones shipped with RAIL, which `pz_data_challenge` installs.

## Configuration

In [1]:
from rail.utils.path_utils import find_rail_file

import znn  # run from the repo root so this picks up the local source

TASKSET = 1
SIMS = ["cardinal", "flagship"]
SCENARIOS = ["1yr", "10yr"]

DATA_DIR = "../pz_data_challenge/scripts/public"
# Starting point of F: "popcosmos" (the pre-trained weights shipped with znn, nothing frozen) or None (from scratch).
# Each choice writes to its own directory, so runs with and without pre-training never overwrite each other.
PRETRAINING = None
NU = znn.DEFAULT_NU  # degrees of freedom of the Student-t G learns
# Method version, appended to OUT_DIR so runs of different versions never overwrite each other ("" for none). It
# starts with "build_model_res_se_log1pz", the name the residual SE model (znn.build_model) had when it was added,
# which keeps its runs apart from the old model's, then the Student-t moment network and its NU
RUN = f"build_model_res_se_log1pz_moment_t{NU:g}"
# where the runs are saved (models, submission files, cached predictions); run_challenge.py sets it to its
# --output-root
OUTPUT_ROOT = "outputs"
OUT_DIR = (f"{OUTPUT_ROOT}/znn" if PRETRAINING is None else f"{OUTPUT_ROOT}/znn_pretrained_{PRETRAINING}") + (f"_{RUN}" if RUN else "")  # submission files for all task sets; the contents of the submission tarball

# Filter curves shipped with RAIL; the challenge's catalogs.yaml ("cardinal_roman_rubin") maps the
# catalogue columns to these DC2LSST_* and roman_* filters
FILTER_DIR = find_rail_file("examples_data/estimation_data/data/FILTER")

# band -> (catalogue magnitude column, filter curve file), ordered by wavelength
BANDS = {
    "u": ("mag_u_lsst", f"{FILTER_DIR}/DC2LSST_u.res"),
    "g": ("mag_g_lsst", f"{FILTER_DIR}/DC2LSST_g.res"),
    "r": ("mag_r_lsst", f"{FILTER_DIR}/DC2LSST_r.res"),
    "i": ("mag_i_lsst", f"{FILTER_DIR}/DC2LSST_i.res"),
    "z": ("mag_z_lsst", f"{FILTER_DIR}/DC2LSST_z.res"),
    "y": ("mag_y_lsst", f"{FILTER_DIR}/DC2LSST_y.res"),
    "Y": ("mag_Y_roman", f"{FILTER_DIR}/roman_Y106.res"),
    "J": ("mag_J_roman", f"{FILTER_DIR}/roman_J129.res"),
    "H": ("mag_H_roman", f"{FILTER_DIR}/roman_H158.res"),
}
REF_BAND = "i"  # magnitudes are normalized by this band
FEATURE_CONFIG = znn.DEFAULT_FEATURE_CONFIG

VAL_FRAC = 0.2  # fraction of the training set held out for evaluation (in pz_challenge_results.ipynb, which must match)
EPOCHS = 100  # max epochs of F and of G (early stopping usually ends sooner)
SEED = 42

I0000 00:00:1793224627.991373 1579929 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import os

import numpy as np
from sklearn.model_selection import train_test_split

os.makedirs(OUT_DIR, exist_ok=True)
print("znn from", znn.__file__)

znn from /global/u2/j/jaimerz/UCL/mlpz_cnn_deepset/znn/__init__.py


## Reference redshifts

In [3]:
def reference_redshifts(train):
    """Reference redshift per training object, and the held-out subsets to score."""
    z_ref = train["redshift"].to_numpy()
    assert np.isfinite(z_ref).all()
    return z_ref, {"held-out": np.ones(len(train), dtype=bool)}

## Run all configurations

For each configuration whose model file is not in `OUT_DIR` yet: train on the fit split and save the model file. The held-out split is left out of training so `pz_challenge_results.ipynb` can score on it.

In [4]:
for SIM in SIMS:
    for SCENARIO in SCENARIOS:
        config = f"{SIM}_{SCENARIO}"
        print(f"\n========== task set {TASKSET}: {config} ==========")
        model_file = os.path.join(OUT_DIR, f"pz_challenge_taskset_{TASKSET}_{SIM}_pz_model_{SCENARIO}.pkl")
        if os.path.exists(model_file):
            print("found", model_file, "(delete it to retrain)")
            continue

        train_file = os.path.join(DATA_DIR, f"pz_challenge_taskset_{TASKSET}_{SIM}_training_{SCENARIO}.hdf5")
        train = znn.read_catalog(train_file)
        z_ref, _ = reference_redshifts(train)

        # the held-out split is not used here, only kept out of training (pz_challenge_results.ipynb scores on it)
        idx_fit, idx_val = train_test_split(np.arange(len(train)), test_size=VAL_FRAC, random_state=SEED)
        model, _ = znn.train_model(
            znn.build_model, train.iloc[idx_fit], BANDS, REF_BAND, FEATURE_CONFIG, Y=z_ref[idx_fit],
            nu=NU, EPOCHS=EPOCHS, random_state=SEED, pretraining=PRETRAINING,
        )

        # subtask 2 submission file: the trained model
        znn.save_model_file(model_file, model, BANDS, REF_BAND, FEATURE_CONFIG)
        print("saved", model_file)


========== task set 1: cardinal_1yr ==========


I0000 00:00:1793224723.532905 1579929 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38478 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:03:00.0, compute capability: 8.0
I0000 00:00:1793224723.546707 1579929 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 38478 MB memory:  -> device: 1, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:41:00.0, compute capability: 8.0
I0000 00:00:1793224723.557253 1579929 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 38478 MB memory:  -> device: 2, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:82:00.0, compute capability: 8.0
I0000 00:00:1793224723.567970 1579929 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 38478 MB memory:  -> device: 3, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:c1:00.0, compute capability: 8.0



=== Base model, on all objects with the photometry as given ===
Epoch 1/100


I0000 00:00:1793224727.038210 1580436 service.cc:153] XLA service 0x7f003c035930 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1793224727.038241 1580436 service.cc:161]   StreamExecutor [0]: NVIDIA A100-SXM4-40GB, Compute Capability 8.0 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1793224727.038250 1580436 service.cc:161]   StreamExecutor [1]: NVIDIA A100-SXM4-40GB, Compute Capability 8.0 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1793224727.038253 1580436 service.cc:161]   StreamExecutor [2]: NVIDIA A100-SXM4-40GB, Compute Capability 8.0 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1793224727.038256 1580436 service.cc:161]   StreamExecutor [3]: NVIDIA A100-SXM4-40GB, Compute Capability 8.0 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1793224727.072571 1580436 dump_mlir_util.cc:269] disabling MLIR crash repro

250/250 - 32s - 128ms/step - loss: 0.0015 - val_loss: 3.0432e-04 - learning_rate: 0.0010
Epoch 2/100
250/250 - 1s - 3ms/step - loss: 5.9338e-04 - val_loss: 2.8350e-04 - learning_rate: 0.0010
Epoch 3/100
250/250 - 1s - 3ms/step - loss: 4.1976e-04 - val_loss: 2.3644e-04 - learning_rate: 0.0010
Epoch 4/100
250/250 - 1s - 3ms/step - loss: 3.6966e-04 - val_loss: 2.0053e-04 - learning_rate: 0.0010
Epoch 5/100
250/250 - 1s - 3ms/step - loss: 3.6648e-04 - val_loss: 2.0510e-04 - learning_rate: 0.0010
Epoch 6/100
250/250 - 1s - 3ms/step - loss: 3.4016e-04 - val_loss: 2.8058e-04 - learning_rate: 0.0010
Epoch 7/100

Epoch 7: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
250/250 - 1s - 3ms/step - loss: 3.5708e-04 - val_loss: 2.2845e-04 - learning_rate: 0.0010
Epoch 8/100
250/250 - 1s - 3ms/step - loss: 3.1456e-04 - val_loss: 1.6724e-04 - learning_rate: 5.0000e-04
Epoch 9/100
250/250 - 1s - 3ms/step - loss: 3.0394e-04 - val_loss: 1.8353e-04 - learning_rate: 5.0000e-04
Epoch 10/1

I0000 00:00:1793224779.199524 1580435 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_33625__.38
I0000 00:00:1793224782.117643 1580432 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_33625__.38
I0000 00:00:1793224783.951691 1580432 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1793224788.314703 1580432 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1793224790.713116 1580432 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a g

... (85 outputs left out)

282/282 - 7s - 26ms/step - loss: 3.8076e-04 - val_loss: 1.0437e-04
Epoch 2/10
282/282 - 1s - 3ms/step - loss: 3.7280e-04 - val_loss: 9.2076e-05
Epoch 3/10
282/282 - 1s - 3ms/step - loss: 3.7019e-04 - val_loss: 9.0811e-05
Epoch 4/10
282/282 - 1s - 3ms/step - loss: 3.7289e-04 - val_loss: 8.8807e-05
Epoch 5/10
282/282 - 1s - 3ms/step - loss: 3.6958e-04 - val_loss: 8.8900e-05
Epoch 6/10
282/282 - 1s - 3ms/step - loss: 3.7204e-04 - val_loss: 8.2118e-05
Epoch 7/10
282/282 - 1s - 3ms/step - loss: 3.6698e-04 - val_loss: 9.0199e-05
Epoch 8/10
282/282 - 1s - 3ms/step - loss: 3.7150e-04 - val_loss: 8.7739e-05
Epoch 9/10
282/282 - 1s - 3ms/step - loss: 3.6811e-04 - val_loss: 8.3907e-05
Epoch 10/10
282/282 - 1s - 3ms/step - loss: 3.6337e-04 - val_loss: 8.9066e-05

--- Member 10/10 (its own noise realization) ---
Epoch 1/10


I0000 00:00:1793225497.124968 1580432 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_599031__.38
I0000 00:00:1793225499.552471 1580432 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_599031__.38


282/282 - 7s - 26ms/step - loss: 3.8201e-04 - val_loss: 1.2547e-04
Epoch 2/10
282/282 - 1s - 3ms/step - loss: 3.7716e-04 - val_loss: 9.8214e-05
Epoch 3/10
282/282 - 1s - 3ms/step - loss: 3.7417e-04 - val_loss: 1.1074e-04
Epoch 4/10
282/282 - 1s - 3ms/step - loss: 3.7916e-04 - val_loss: 1.0693e-04
Epoch 5/10
282/282 - 1s - 3ms/step - loss: 3.6709e-04 - val_loss: 9.6700e-05
Epoch 6/10
282/282 - 1s - 3ms/step - loss: 3.6604e-04 - val_loss: 9.2622e-05
Epoch 7/10
282/282 - 1s - 3ms/step - loss: 3.6698e-04 - val_loss: 9.6776e-05
Epoch 8/10
282/282 - 1s - 3ms/step - loss: 3.6954e-04 - val_loss: 8.8206e-05
Epoch 9/10
282/282 - 1s - 3ms/step - loss: 3.6125e-04 - val_loss: 9.2636e-05
Epoch 10/10
282/282 - 1s - 3ms/step - loss: 3.7333e-04 - val_loss: 9.6398e-05


saved outputs/znn_pretrained_popcosmos_noisy_kfold_frombase/pz_challenge_taskset_1_flagship_pz_model_10yr.pkl
